# count how much sea ice we are removing

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import sys
import dask.dataframe as dd
from dask.distributed import wait

from scipy.ndimage import binary_closing as binary_closing
from scipy.ndimage import binary_opening as binary_opening
from scipy.ndimage import label as scipy_label
from scipy.ndimage import sum as scipy_sum
from skimage.measure import regionprops


import os
import seaborn as sns
import intake
from scipy.stats import linregress
from scipy import stats
from dask import delayed
from IPython.display import display, HTML
import imageio
#from joblib import Parallel, delayed
import matplotlib.dates as mdates
import intake
from DV8_extremes import *
#DV8_functions(1) packages
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.animation import FFMpegWriter
from matplotlib import animation, rc
from IPython.display import HTML
#my functions:

#from dask_image.ndmorph import binary_opening, binary_closing 


import matplotlib.colors as mcolors

from tempfile import TemporaryDirectory
from getpass import getuser
from pathlib import Path
import dask
from dask.distributed import Client, LocalCluster
import bokeh
import subprocess
import re

import warnings
#warnings.filterwarnings('ignore')
scratch_dir = Path('/scratch') / getuser()[0] / getuser()

/home/b/b382616/.conda/envs/super_trooper/lib/python3.14/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [2]:


dask.config.config.get('distributed').get('dashboard').update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})
cluster = LocalCluster(n_workers=32,threads_per_worker=1)
client = Client(cluster)
client

/home/b/b382616/.conda/envs/super_trooper/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 34861 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/b382616/levante-spawner-advanced//proxy/34861/status,
Dashboard: /user/b382616/levante-spawner-advanced//proxy/34861/status,Workers: 32
Total threads: 32,Total memory: 250.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36551,Workers: 0
Dashboard: /user/b382616/levante-spawner-advanced//proxy/34861/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:39041,Total threads: 1
Dashboard: /user/b382616/levante-spawner-advanced//proxy/42665/status,Memory: 7.81 GiB
Nanny: tcp://127.0.0.1:39835,


# LOAD DATA


In [3]:

def lon_180w_180e(ds_sice: xr.DataArray):
    ds_sice = (
        ds_sice
        .assign_coords(lon=lambda x: (x.lon - 180) % 360 - 180)
        .sortby('lon'))

    return ds_sice

In [4]:
#SST

# OSTIA
filename='/scratch/b/b382615/mhws/ostia.zarr' 
sst = xr.open_zarr(str(filename), chunks={'time':150, 'lat':-1, 'lon':-1}).sst
sst=sst.sel(lat=slice(-50, 70))
sst=sst-273.15

# ICON HISTORICAL
cat = intake.open_catalog("https://raw.githubusercontent.com/eerie-project/intake_catalogues/main/eerie.yaml")
dat = cat['dkrz.disk.model-output.icon-esm-er.hist-1950.v20240618.ocean.gr025']
sst_ic = dat['2d_daily_mean'](chunks={}).to_dask().to.isel(depth=0).drop_vars('depth').chunk({'time':150, 'lat':-1, 'lon':-1})
sst_ic=sst_ic.sel(lat=slice(-50, 70))


# IFS FESOM
cat = intake.open_catalog("https://raw.githubusercontent.com/eerie-project/intake_catalogues/main/eerie.yaml")
dat = cat['dkrz.disk.model-output.ifs-fesom2-sr.hist-1950.v20240304.ocean.gr025']#.icon-esm-er.hist-1950.v20240618.ocean.gr025']
sst_f = dat['2D_daily_avg_1950-2014'].to_dask().avg_tos.isel(depth=0).drop_vars('depth').chunk({'time':150, 'lat':-1, 'lon':-1})
sst_f=sst_f.sel(lat=slice(-50, 70))



In [5]:
# HADGEM LL
sst_LL = xr.open_dataset(
    '/home/b/b382616/scratch/mhws/hadgem/input_data/LL/toscon_nemo_u-da156_1d_19700101-20150101_grid-025.nc',
    chunks={'time': 150, 'lat': -1, 'lon': -1}  # Chunk by year, full spatial dimensions
)
sst_LL=sst_LL['toscon'].sel(lat=slice(-50, 70))


# HADGEM MM
sst_MM = xr.open_dataset(
    '/home/b/b382616/scratch/mhws/hadgem/input_data/MM/toscon_nemo_u-dc396_1d_19700101-20150101_grid-025.nc',
    chunks={'time': 150, 'lat': -1, 'lon': -1}  # Chunk by year, full spatial dimensions
)
sst_MM=sst_MM['toscon'].sel(lat=slice(-50, 70))


# HADGEM HH
sst_HH = xr.open_dataset(
    '/home/b/b382616/scratch/mhws/hadgem/input_data/HH/toscon_nemo_u-di356_1d_19700101-20150101_grid-025.nc',
    chunks={'time': 150, 'lat': -1, 'lon': -1}  # Chunk by year, full spatial dimensions
)
sst_HH=sst_HH['toscon'].sel(lat=slice(-50, 70))

            


/home/b/b382616/.conda/envs/super_trooper/lib/python3.14/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'toscon' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/home/b/b382616/.conda/envs/super_trooper/lib/python3.14/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'toscon' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/home/b/b382616/.conda/envs/super_trooper/lib/python3.14/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'toscon' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


# what value represents sea ice in each dataset

In [11]:
models_sst = {
    'OSTIA': sst,     
    'ICON': sst_ic,      
    'IFS-FESOM': sst_f,  
    'HadGEM3-LL': sst_LL,
    'HadGEM3-MM': sst_MM,
    'HadGEM3-HH': sst_HH
}

In [12]:
# Focus on a cold spot: 60N, 160E (Sea of Okhotsk) in mid-winter
test_lat = 60
test_lon = 160
test_time = "2000-02-15"

print(f"{'Model':<12} | {'SST Value':<10}")
print("-" * 25)

for name, data in models_sst.items():
    try:
        # Select the nearest point and time
        val = data.sel(lat=test_lat, lon=test_lon, time=test_time, method='nearest').values
        print(f"{name:<12} | {val.item():.4f}")
    except Exception as e:
        print(f"{name:<12} | Error: {e}")

Model        | SST Value 
-------------------------
OSTIA        | -1.8000
ICON         | -1.7992
IFS-FESOM    | -1.6964
HadGEM3-LL   | -1.4139
HadGEM3-MM   | -1.7244
HadGEM3-HH   | -1.7720


In [13]:


# We'll check how many pixels are within a tiny margin of the values you found
check_values = {
    'OSTIA': -1.8000,
    'ICON': -1.7992,
    'IFS-FESOM': -1.6964,
    'HadGEM3-LL': -1.4139,
    'HadGEM3-MM': -1.7244,
    'HadGEM3-HH': -1.7720
}

print(f"{'Model':<12} | {'Target':<8} | {'Count of values <= Target + 0.01'}")
print("-" * 55)

for name, target in check_values.items():
    ds = models_sst[name].sel(lat=slice(50, 70))
    # Count how many points are at or below this 'floor'
    count = (ds <= (target + 0.01)).sum().compute().item()
    print(f"{name:<12} | {target:<8.4f} | {count}")

Model        | Target   | Count of values <= Target + 0.01
-------------------------------------------------------
OSTIA        | -1.8000  | 86733989
ICON         | -1.7992  | 147236706
IFS-FESOM    | -1.6964  | 117634742
HadGEM3-LL   | -1.4139  | 95712749
HadGEM3-MM   | -1.7244  | 31116695
HadGEM3-HH   | -1.7720  | 17884744


# Percentage of sea-ice values we are removing

In [6]:
# Create a dictionary to store the counts
sea_ice_counts = {}

# OSTIA
sea_ice_counts['OSTIA'] = (sst <= -1.7).sum().compute().item()

# ICON
sea_ice_counts['ICON'] = (sst_ic <= -1.7).sum().compute().item()

# IFS-FESOM
sea_ice_counts['IFS-FESOM'] = (sst_f <= -1.7).sum().compute().item()

# HADGEM LL
sea_ice_counts['HadGEM-LL'] = (sst_LL <= -1.7).sum().compute().item()

# HADGEM MM
sea_ice_counts['HadGEM-MM'] = (sst_MM <= -1.7).sum().compute().item()

# HADGEM HH
sea_ice_counts['HadGEM-HH'] = (sst_HH <= -1.7).sum().compute().item()

# Print the results
for model, count in sea_ice_counts.items():
    print(f"{model}: {count} sea ice gridcell-timesteps")

OSTIA: 100656161 sea ice gridcell-timesteps
ICON: 158421476 sea ice gridcell-timesteps
IFS-FESOM: 113570655 sea ice gridcell-timesteps
HadGEM-LL: 31085126 sea ice gridcell-timesteps
HadGEM-MM: 37321190 sea ice gridcell-timesteps
HadGEM-HH: 41841302 sea ice gridcell-timesteps


In [7]:
sea_ice_counts

{'OSTIA': 100656161,
 'ICON': 158421476,
 'IFS-FESOM': 113570655,
 'HadGEM-LL': 31085126,
 'HadGEM-MM': 37321190,
 'HadGEM-HH': 41841302}

In [ ]:
# Assuming you still have the sst variables and sea_ice_counts dictionary in memory
percentages = {}

# We use .count() because it only counts non-NaN values (ocean points)
# This ensures we aren't including land in our "Total" percentage denominator.

# OSTIA
total_os = sst.count().compute().item()
percentages['OSTIA'] = (sea_ice_counts['OSTIA'] / total_os) * 100

# ICON
total_ic = sst_ic.count().compute().item()
percentages['ICON'] = (sea_ice_counts['ICON'] / total_ic) * 100

# IFS-FESOM
total_f = sst_f.count().compute().item()
percentages['IFS-FESOM'] = (sea_ice_counts['IFS-FESOM'] / total_f) * 100

# HADGEM LL
total_LL = sst_LL.count().compute().item()
percentages['HadGEM-LL'] = (sea_ice_counts['HadGEM-LL'] / total_LL) * 100

# HADGEM MM
total_MM = sst_MM.count().compute().item()
percentages['HadGEM-MM'] = (sea_ice_counts['HadGEM-MM'] / total_MM) * 100

# HADGEM HH
total_HH = sst_HH.count().compute().item()
percentages['HadGEM-HH'] = (sea_ice_counts['HadGEM-HH'] / total_HH) * 100


In [10]:
# Create a mapping of the model names to the total variables you already calculated
totals_mapping = {
    'OSTIA': total_os,
    'ICON': total_ic,
    'IFS-FESOM': total_f,
    'HadGEM-LL': total_LL,
    'HadGEM-MM': total_MM,
    'HadGEM-HH': total_HH
}

print(f"{'Model':<15} | {'Ice Count':<12} | {'Total Ocean':<15} | {'Percentage'}")
print("-" * 65)

for model in percentages.keys():
    ice = sea_ice_counts[model]
    total = totals_mapping[model]
    pct = percentages[model]
    print(f"{model:<15} | {ice:<12} | {total:<15} | {pct:.2f}%")

Model           | Ice Count    | Total Ocean     | Percentage
-----------------------------------------------------------------
OSTIA           | 100656161    | 7082047341      | 1.42%
ICON            | 158421476    | 11261638314     | 1.41%
IFS-FESOM       | 113570655    | 11216601637     | 1.01%
HadGEM-LL       | 31085126     | 7765303252      | 0.40%
HadGEM-MM       | 37321190     | 7765303252      | 0.48%
HadGEM-HH       | 41841302     | 7765303252      | 0.54%
